In [0]:
# label = 1 → cliente comprou o produto
# label = 0 → cliente não comprou o produto

In [0]:
# feature_end_date = 2018-07-01
# target_end_date  = 2018-08-01

In [0]:
from pyspark.sql.functions import lit

In [0]:
customer_features = (spark.table("ml_training_dev.feature.customers")
                    .withColumnRenamed("total_orders", "total_orders_customer")
                    .withColumnRenamed("total_spent", "total_spent_customer")
                   
)


In [0]:
interaction_features = spark.table("ml_training_dev.feature.customer_product_interactions")

"interaction_total_spent",
    "interaction_avg_price",
    "interaction_avg_freight",
    
product_features = spark.table("ml_training_dev.feature.products").withColumnRenamed("avg_price", "avg_price_products")

In [0]:
positive = (
    interaction_features
    .select(
        "customer_unique_id",
        "product_id"
    )
    .withColumn("label",lit(1))
)

In [0]:
training = (
    interaction_features
    .join(
        customer_features,
        on="customer_unique_id",
        how="left"
    )
    .join(
        product_features,
        on="product_id",
        how="left"
    )
)

In [0]:
feature_cols = [
    "total_orders",
    "total_items",
    "total_spent",
    "avg_order_value",
    "unique_products",
    "unique_categories",

    "avg_price",
    "avg_freight_value",
    "total_product_orders",
    "unique_customers",
    "total_revenue",
    "avg_rating",

    "purchases",
    "interaction_total_spent",
    "interaction_avg_price",
    "interaction_avg_freight"
]

In [0]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import LogisticRegression

assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features"
)

train_vector = assembler.transform(training)
train_vector.display()

In [0]:
lr = LogisticRegression(
    featuresCol="features",
    labelCol="label",
    predictionCol="prediction",
    probabilityCol="probability"
)

model = lr.fit(train_vector)

In [0]:
predictions = model.transform(test_vector)

In [0]:
# P(buy product | customer)

In [0]:
recommendations = (
    predictions
    .withColumn(
        "probability_buy",
        F.col("probability")[1]
    )
    .withColumn(
        "rank",
        F.row_number().over(window_customer)
    )
    .filter(F.col("rank") <= 10)
)